In [1]:
# %pip install -q langchain-ollama

In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.2:1b")  
llm.invoke("What is the capital of France?") # 프롬프트 : LLM호출 

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T02:12:59.4234529Z', 'done': True, 'done_reason': 'stop', 'total_duration': 539015000, 'load_duration': 90324100, 'prompt_eval_count': 32, 'prompt_eval_duration': 214484100, 'eval_count': 8, 'eval_duration': 221474000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce4f7-eb82-73a3-87bd-3e38d983c1d0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser  # output parser

prompt_template = PromptTemplate(
    template = "What is the capital of {country}?",    # Return the name of the city. (수도만 나오게끔 하려면)
    input_variables=["country"],
)

prompt = prompt_template.invoke({"country":"France"})  
ai_message = llm.invoke(prompt)
ai_message

AIMessage(content='The capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T02:12:59.63248Z', 'done': True, 'done_reason': 'stop', 'total_duration': 186387400, 'load_duration': 111902100, 'prompt_eval_count': 32, 'prompt_eval_duration': 9150900, 'eval_count': 8, 'eval_duration': 55505300, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce4f7-edb5-7920-bd2a-9b7ffbf84cb9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 32, 'output_tokens': 8, 'total_tokens': 40})

In [4]:
output_parser = StrOutputParser()  # output parser 정
answer = output_parser.invoke(llm.invoke(prompt))
answer

'The capital of France is Paris.'

In [5]:
# ai_message와 answer의 차이점

# ai_message는 AIMessage 클래스 (BaseMessage)로 나온다 
# asnwer 은 string형식으로 나온다

# base 보다는 string이 더 편리 -> 프론트로 전달한다고 가정 : 프론트에는 baseMessage가 없어 parsing불가 -> string으로..

In [6]:
from langchain_core.output_parsers import JsonOutputParser

country_detail_prompt = PromptTemplate(
    template = """Give following information about {country}:
    - Capital
    - Population
    - Language
    - Currency

    return it in JSON format. and return the JSON dictionary only
    """,
    input_variables=["country"],
)

country_detail_prompt = country_detail_prompt.invoke({"country" : "France"})

output_parser_json = JsonOutputParser()
output_parser_json.invoke(llm.invoke(country_detail_prompt))  # 에러 발생

OutputParserException: Invalid json output: {
    "capital": "Paris",
    "population": 67.2 million,
    "language": "French",
    "currency": "Euro"
}
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 

In [7]:
country_detail_prompt

StringPromptValue(text='Give following information about France:\n    - Capital\n    - Population\n    - Language\n    - Currency\n\n    return it in JSON format. and return the JSON dictionary only\n    ')

In [8]:
error_json = llm.invoke(country_detail_prompt)
error_json.content

AIMessage(content='```\n{\n    "capital": "Paris",\n    "population": 67.2,\n    "language": "French",\n    "currency": "Euro"\n}\n```', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-13T02:10:50.7399449Z', 'done': True, 'done_reason': 'stop', 'total_duration': 765033100, 'load_duration': 96791300, 'prompt_eval_count': 62, 'prompt_eval_duration': 192789600, 'eval_count': 36, 'eval_duration': 399619000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019ce4f5-f3f6-75e1-8631-737a218cc231-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 36, 'total_tokens': 98})

In [12]:
# 사실상 error_json은 str 타입이다 -> str인데 JSONOutput을 활용해서 에러가 나는 것
# type(error_json.content)  
type(ai_message.content)  # str
# type(error_json.content)

str